In [1]:
f_unmaching =None
f_output = None

## 指定パス内にある全アルバム・フォルダを取得する。

In [2]:
from pathlib import Path
def get_album_names(folder_base_path):
    """アルバム名のリストを取得する関数

    Args:
        folder_base_path (str): フォルダのベースパス

    Returns:
        list: アルバム名のリスト
    """
    # 1. ベースパスの設定（~ をフルパスに展開）
    folder_path = Path(folder_base_path).expanduser()

    # 2. ディレクトリ内を走査し、フォルダ（ディレクトリ）のみをリストに追加
    album_names = [f.name for f in folder_path.iterdir() if f.is_dir()]
    return album_names


In [3]:
import os
def get_fig_and_json_files(album_name_path):
    """画像ファイルリストと画像情報(json)ファイルリストを取得する関数

    Args:
        album_name_path (pathlib.Path): アルバムのパス名

    Returns:
        tuple: 画像ファイル(list), 画像情報(json)ファイル(list)
    """
    fig_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff']
    json_extensions = ['.json']
    fig_files, json_files = [], []

    for item in album_name_path.iterdir():
        if item.is_file():
            name, ext = os.path.splitext(item.name)
            if ext.lower() in fig_extensions:
                fig_files.append(item)
            elif ext.lower() in json_extensions :
                json_files.append(item)

    return fig_files, json_files



In [4]:
import json
def find_figname_from_json(json_path):
    """JSONのtitleから画像ファイル名を取得する関数

    Args:
        json_path (path): JSONファイルのパス

    Returns:
        str: 画像ファイル名
    """

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        # JSONの中にある "title" が、実際の画像ファイル名であることが多い
        original_name = data.get('title')
        return original_name

In [5]:
import re
import os

def extract_n_and_clean_filename(filename):
    """ファイル名から (n) を探し、nと除去後のファイル名を返す。
     nは1〜100まで。存在しない場合は n=0 を返す。

    Args:
        filename (str): ファイル名文字列

    Returns:
        tuple: (除去後の拡張子なしファイル名,除去後のファイル名, n)
    """
 
    # # 正規表現の解説:
    # \s*     : 数字の前のスペース（もしあれば）を対象に含む
    # \(      : 開始カッコ
    # ([1-9][0-9]?|100) : 1-99 または 100 の数字をキャプチャ（グループ1）
    # \)      : 閉じカッコ
    # 1〜100までの (n) を探すパターン
    pattern = r'\s*\(([1-9][0-9]?|100)\)'
    
    match = re.search(pattern, filename)
    
    if match:
        n = int(match.group(1))
        # カッコ部分を除去したフルファイル名を作成
        full_name = re.sub(pattern, "", filename)
    else:
        n = 0
        full_name = filename

    # 拡張子とそれ以外を分割
    # os.path.splitext("image.jpg") -> ("image", ".jpg")
    name_part, extension = os.path.splitext(full_name)
    
    return name_part, full_name, n



In [6]:

def get_fig2json_db(fig_files, json_files):
    """写真リストと写真情報(JSON)リストのマッピングを生成する関数

    Args:
        fig_files (list): 写真リスト_
        json_files (list): 写真情報(JSON)リスト_

    Returns:
        map: 写真ファイルと写真情報(JSON)ファイルのマッピング
    """
    num_not_matched = 0
    fig2json_db = {}
    for f in fig_files:
        fname, full_name, fn = extract_n_and_clean_filename(f.name)
        l =len(fname)
        # print(f"FIG_: {f.name}, name: {fname}, fn: {fn}")

        is_matched = False
        for j in json_files:
            jname, jcur_name, jn = extract_n_and_clean_filename(j.name)
            jfull_name = find_figname_from_json(j)
            if jfull_name is None: continue
            jfull_name = jfull_name[0:l]
            if fname == jfull_name and fn == jn:
                fig2json_db[f] = j
                is_matched = True
                # print(f"JSON: {j.name}, name: {jfull_name}, jn: {jn}")
                break

        if not is_matched:
            print(f"No matching JSON file found for FIG: {f.name}",file=f_unmaching)
            num_not_matched += 1

    print(f"Total unmatched FIG files: {num_not_matched}/{len(fig_files)}")
    return fig2json_db



In [7]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import json

def get_photo_taken_datetime(json_file_path):
    """写真情報(JSON)ファイルから写真撮影日時を取得する関数

    Args:
        json_file_path (path):写真情報(JSON)ファイルのパス

    Returns:
        datetime: 写真撮影日時 (datetimeオブジェクト)
    """

    # Helper function to convert timestamp to datetime
    def convert_timestamp(timestamp):
        return datetime.fromtimestamp(int(timestamp), tz=timezone.utc)

    with json_file_path.open('r') as json_file:
        json_data = json.load(json_file)
    
    # Extract relevant metadata
    creation_time = json_data.get('creationTime', {}).get('timestamp')
    photo_taken_time = json_data.get('photoTakenTime', {}).get('timestamp')
    
    # Convert timestamp to datetime
    # Googleフォトのライブラリに作成（アップロード）された日
    creation_datetime = convert_timestamp(creation_time) if creation_time else None
    # 写真が実際に撮影された日時です
    photo_taken_datetime = convert_timestamp(photo_taken_time) if photo_taken_time else None

    # print(f"Creation Time: {creation_datetime}")
    # print(f"Photo Taken Time: {photo_taken_datetime}")

    # timedelta(hours=9) を足したことで、Googleフォトのシステム時間（UTC）から、
    # 日本時間（JST）へと正確に変換される。
    if photo_taken_datetime:
        photo_taken_datetime += timedelta(hours=9)
        
    return  photo_taken_datetime


## フォルダ内のアルバムを取得する

In [8]:
import os

folder_base_path = "~/SharedFolder/junko/Google フォト/"
album_names = get_album_names(folder_base_path)

# 結果の確認
print(album_names)
print(f"Total albums: {len(album_names)}")

['羽生結弦', '2017 年の写真', '2020 年の写真', 'トリノとミラノへの旅行', '2016 年の写真', 'ドボ＆コナ', '北海道への旅行', '2026 年の写真', '2025 年の写真', 'イタリアへの旅行', '2014-08-05', 'スペイン＆フランス\u3000Voyage', 'レシピ', '石垣島＆西表島', 'Test', '2017\u3000スペイン＆フレンチバスク＆France南西地方＆パリ', '三兄弟', 'フランスへの旅行', '2023 年の写真', '2016年北イタリア、チンクエテッレ＆トスカーナ', '2014 年の写真', '2015 年の写真', '2018 年の写真', '🇹🇭バンコクの旅', '2022 年の写真', '姫路城', '２０１５年南イタリア、プーリア＆アマルフィの旅', 'France南西部の旅2014', '無題(1)', 'コート', '無題', '2026年1月22日〜30日\u3000ベトナム\u3000ホーチミンの旅', '2019 年の写真', '我が家の子供たち', '我が家の癒やし❤', '2021 年の写真', 'アーカイブ', '京都市', '2024 年の写真', 'ゆづ', '広島＆厳島神社', '金沢\u3000avec 八木ちゃん']
Total albums: 42


## 各アルバム内の画像ファイル(.jpg) を取得する。

In [9]:
from datetime import timedelta

# '我が家の子供たち'
# '2014 年の写真'
# 'France南西部の旅2014'
# '2019 年の写真'
# 'フランスへの旅行'
# '2015 年の写真'

folder_path = Path(folder_base_path).expanduser()

# for album_no, album_name in enumerate(["Test"]):
# for album_no, album_name in enumerate(['我が家の子供たち']):
# for album_no, album_name in enumerate(['2014 年の写真']):
for album_no, album_name in enumerate(['我が家の子供たち', '我が家の癒やし❤', '2021 年の写真', 'アーカイブ', '京都市', '2024 年の写真', 'ゆづ', '広島＆厳島神社', '金沢\u3000avec 八木ちゃん']):
# for album_no, album_name in enumerate(album_names):

    log_folder_path = "./260502_log_photo"
    f_output = open( f"{log_folder_path}/{album_name}_output.txt","w")
    f_unmaching = open(f"{log_folder_path}/{album_name}_unmatching.txt", 'w')
    f_output.write(f"Album No: {album_no}, Album Name: {album_name}\n\n")
    f_unmaching.write(f"Album No: {album_no}, Album Name: {album_name}\n\n")

    album_name_path = folder_path / album_name
    fig_files, json_files = get_fig_and_json_files(album_name_path)
    
    print(f"Total FIG files: {len(fig_files)}")
    print(f"Total JSON files: {len(json_files)}")

    fig2json_db = get_fig2json_db(fig_files, json_files)
    fig_files = json_files = []
        
    num_changed =0
    for f, j in fig2json_db.items():
        print(f"FIG_: {f.name}",file=f_output ) 
        print(f"JSON: {j.name}",file=f_output )
        
        num_changed += 1
        photo_taken_datetime = get_photo_taken_datetime(j)  
        print(f"Photo Taken Time: [{photo_taken_datetime}]",file=f_output)
        print(file=f_output)

        # datetimeオブジェクトを「エポック秒（Unix時間）」に変換
        # OSが時間を扱うための数値形式にする必要があります
        timestamp = photo_taken_datetime.timestamp()
        
        # ファイルの「アクセス日時」と「更新日時」を書き換え
        # (アクセス日時, 更新日時) の順で指定する
        os.utime(f, (timestamp, timestamp))
        # print( f"更新完了: {f} -> {photo_taken_datetime}")
    
    print(f"Total updated files: {num_changed}")          

    f_unmaching.close()
    f_output.close()


Total FIG files: 661
Total JSON files: 525
Total unmatched FIG files: 138/661
Total updated files: 523
Total FIG files: 23
Total JSON files: 24
Total unmatched FIG files: 0/23
Total updated files: 23
Total FIG files: 653
Total JSON files: 653
Total unmatched FIG files: 0/653
Total updated files: 653
Total FIG files: 12
Total JSON files: 12
Total unmatched FIG files: 0/12
Total updated files: 12
Total FIG files: 34
Total JSON files: 35
Total unmatched FIG files: 0/34
Total updated files: 34
Total FIG files: 1287
Total JSON files: 1289
Total unmatched FIG files: 0/1287
Total updated files: 1287
Total FIG files: 11
Total JSON files: 12
Total unmatched FIG files: 0/11
Total updated files: 11
Total FIG files: 115
Total JSON files: 116
Total unmatched FIG files: 0/115
Total updated files: 115
Total FIG files: 250
Total JSON files: 251
Total unmatched FIG files: 0/250
Total updated files: 250
